# Visualizacao com t-SNE

Neste notebook vamos carregar as matrizes **Bag of Words** e **TF-IDF** geradas no notebook `introducao_pln` e aplicar o **t-SNE** para reduzir a dimensionalidade a duas dimensoes. Assim conseguimos visualizar como as noticias se agrupam segundo cada representacao.

## O que e t-SNE

O **t-SNE** (*t-distributed Stochastic Neighbor Embedding*) e uma tecnica de reducao de dimensionalidade nao linear, muito usada para visualizacao. Ele tenta colocar pontos parecidos no espacao original proximos no espaco reduzido (em geral 2D), preservando a estrutura local dos dados.

Aqui, cada ponto sera uma noticia. Esperamos que noticias com vocabulario parecido apareçam proximas no grafico.

In [2]:
import pandas as pd
import plotly.express as px
from sklearn.manifold import TSNE

## Carregando os dados

In [3]:
df_bow = pd.read_csv("../dados/bow.csv")
df_tfidf = pd.read_csv("../dados/tfidf.csv")

print(f"BoW:    {df_bow.shape}")
print(f"TF-IDF: {df_tfidf.shape}")
df_bow.head()

BoW:    (2000, 8444)
TF-IDF: (2000, 8441)


,data,titulo,visitas,bow_abafa,bow_abaixo,bow_abalou,bow_abandonar,bow_abdicar,bow_abel,bow_aberta,...,bow_zaneratto,bow_zanni,bow_zanotti,bow_ze,bow_zenit,bow_zerar,bow_zero,bow_zona,bow_zorzi,bow_zubeldia
0,11/4/2026 16:47,STJD indefere pedido de suspensão e Abel Ferre...,141,0,0,0,0,0,3,0,...,0,0,0,0,0,0,0,0,0,0
1,11/4/2026 16:47,Nubank Anuncia Amistoso Imperdível: Palmeiras ...,129,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,11/4/2026 16:47,STJD Recusa Suspensão de Abel Ferreira; Palmei...,105,0,0,0,0,0,3,0,...,0,0,0,0,0,0,0,0,0,0
3,11/4/2026 14:27,Palmeiras encerra treinos para o Dérbi com Vit...,276,0,0,0,0,0,3,0,...,0,0,0,0,0,0,0,0,0,0
4,11/4/2026 14:07,Messi Pode Anunciar Amistoso Histórico em São ...,246,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [6]:
df_tfidf

,abafa,abaixo,abalou,abandonar,abdicar,abel,aberta,abertamente,abertas,aberto,...,zaneratto,zanni,zanotti,ze,zenit,zerar,zero,zona,zorzi,zubeldia
0,0.000000,0.0,0.0,0.0,0.0,0.075833,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.0,0.0,0.0,0.0,0.070553,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.000000,0.0,0.0,0.0,0.0,0.068345,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,0.000000,0.0,0.0,0.0,0.0,0.060562,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1996,0.114391,0.0,0.0,0.0,0.0,0.050720,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1997,0.000000,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1998,0.000000,0.0,0.0,0.0,0.0,0.064445,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Separando metadados e matrizes

As colunas que comecam com `bow_` ou `tfidf_` sao as variaveis numericas. As outras sao metadados (titulo, url, tags, data).

In [7]:
colunas_bow = [c for c in df_bow.columns if c.startswith("bow_")]
colunas_tfidf = [c for c in df_tfidf.columns]

X_bow = df_bow[colunas_bow].values
X_tfidf = df_tfidf[colunas_tfidf].values

titulos = df_bow["titulo"].tolist()

print(f"X_bow:   {X_bow.shape}")
print(f"X_tfidf: {X_tfidf.shape}")

X_bow:   (2000, 8441)
X_tfidf: (2000, 8441)


## Aplicando o t-SNE

Como temos poucas noticias (16), usamos `perplexity` baixo (o valor precisa ser menor que o numero de amostras).

In [8]:
perplexity = 5
random_state = 42

tsne_bow = TSNE(n_components=2, perplexity=perplexity, random_state=random_state, init="random")
embed_bow = tsne_bow.fit_transform(X_bow)

tsne_tfidf = TSNE(n_components=2, perplexity=perplexity, random_state=random_state, init="random")
embed_tfidf = tsne_tfidf.fit_transform(X_tfidf)

print("embed_bow:  ", embed_bow.shape)
print("embed_tfidf:", embed_tfidf.shape)

embed_bow:   (2000, 2)
embed_tfidf: (2000, 2)


## Visualizando lado a lado

In [9]:
def plotar(embed, nome):
    df_plot = pd.DataFrame({
        "x": embed[:, 0],
        "y": embed[:, 1],
        "indice": df_bow.index,
        "titulo": df_bow["titulo"],
        "data": df_bow["data"],
    })
    fig = px.scatter(
        df_plot,
        x="x",
        y="y",
        hover_data={"indice": True, "titulo": True, "data": True, "x": False, "y": False},
        title=f"t-SNE - {nome}",
        width=700,
        height=700,
    )
    fig.write_html(nome+".html")
    fig.show()


plotar(embed_bow, "Bag of Words")
plotar(embed_tfidf, "TF-IDF")